# 01 - Exploratory Data Analysis

This notebook explores the IBM Telco Customer Churn dataset with univariate analysis, bivariate analysis, churn rate breakdowns, Seaborn charts, and Plotly charts.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import plotly.express as px
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT / "src"))

from preprocess import clean_telco_data, load_telco_data

sns.set_theme(style="whitegrid", palette="Set2")
DATA_PATH = PROJECT_ROOT / "data" / "WA_Fn-UseC_-Telco-Customer-Churn.csv"

In [ ]:
raw_df = load_telco_data(DATA_PATH)
df = clean_telco_data(raw_df)
df["ChurnLabel"] = df["Churn"].map({0: "No", 1: "Yes"})

print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")
df.head()

## Data Quality and Target Balance

In [ ]:
missing_summary = (
    df.isna().sum()
    .loc[lambda values: values > 0]
    .sort_values(ascending=False)
    .to_frame("missing_count")
)
missing_summary["missing_rate"] = missing_summary["missing_count"] / len(df)
missing_summary

In [ ]:
churn_counts = df["ChurnLabel"].value_counts().rename_axis("Churn").reset_index(name="customers")
fig = px.pie(
    churn_counts,
    values="customers",
    names="Churn",
    title="Overall Churn Distribution",
    hole=0.45,
)
fig.show()

df["Churn"].mean()

## Univariate Analysis

In [ ]:
categorical_columns = [
    "gender",
    "Partner",
    "Dependents",
    "PhoneService",
    "MultipleLines",
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
    "Contract",
    "PaperlessBilling",
    "PaymentMethod",
]

for column in ["Contract", "InternetService", "PaymentMethod"]:
    plt.figure(figsize=(9, 4))
    sns.countplot(data=df, y=column, order=df[column].value_counts().index)
    plt.title(f"Customer Count by {column}")
    plt.xlabel("Customers")
    plt.ylabel(column)
    plt.tight_layout()
    plt.show()

In [ ]:
numeric_columns = ["tenure", "MonthlyCharges", "TotalCharges"]
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for axis, column in zip(axes, numeric_columns):
    sns.histplot(data=df, x=column, kde=True, ax=axis)
    axis.set_title(f"Distribution of {column}")
plt.tight_layout()
plt.show()

## Churn Rate Breakdowns

In [ ]:
def churn_rate_by(column: str) -> pd.DataFrame:
    summary = (
        df.groupby(column, dropna=False)
        .agg(customers=("Churn", "size"), churn_rate=("Churn", "mean"))
        .reset_index()
        .sort_values("churn_rate", ascending=False)
    )
    summary["churn_rate_pct"] = summary["churn_rate"] * 100
    return summary

contract_churn = churn_rate_by("Contract")
fig = px.bar(
    contract_churn,
    x="Contract",
    y="churn_rate_pct",
    text="churn_rate_pct",
    title="Churn Rate by Contract Type",
    labels={"churn_rate_pct": "Churn rate (%)"},
)
fig.update_traces(texttemplate="%{text:.1f}%", textposition="outside")
fig.show()
contract_churn

In [ ]:
for column in ["InternetService", "PaymentMethod", "OnlineSecurity", "TechSupport"]:
    display(churn_rate_by(column))

## Bivariate Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for axis, column in zip(axes, numeric_columns):
    sns.boxplot(data=df, x="ChurnLabel", y=column, ax=axis)
    axis.set_title(f"{column} by Churn")
plt.tight_layout()
plt.show()

In [ ]:
fig = px.scatter(
    df,
    x="tenure",
    y="MonthlyCharges",
    color="ChurnLabel",
    opacity=0.55,
    title="Tenure vs Monthly Charges by Churn",
)
fig.show()

In [ ]:
plt.figure(figsize=(7, 5))
corr = df[["SeniorCitizen", "tenure", "MonthlyCharges", "TotalCharges", "Churn"]].corr()
sns.heatmap(corr, annot=True, cmap="vlag", center=0, fmt=".2f")
plt.title("Numeric Feature Correlations")
plt.tight_layout()
plt.show()